# Supervised Fine-Tuning & Instruction Tuning

**Companion lesson:** https://ml-viz.vercel.app/courses/fine-tuning-alignment/01-supervised-fine-tuning

A from-scratch, runnable demo of SFT mechanics — prompt-masked cross-entropy, gradient updates on a tiny character-level model, and catastrophic forgetting. Pure NumPy, no API keys.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(0)

## A tiny next-token model

We'll use a single linear layer over one-hot character embeddings — enough to feel the dynamics of SFT without GPU time. Vocabulary is just the lowercase alphabet plus space.

In [ ]:
VOCAB = list('abcdefghijklmnopqrstuvwxyz ')
V = len(VOCAB)
ch2i = {c: i for i, c in enumerate(VOCAB)}

def encode(s):
    return np.array([ch2i[c] for c in s], dtype=int)

def one_hot(ids):
    x = np.zeros((len(ids), V))
    x[np.arange(len(ids)), ids] = 1
    return x

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

W = rng.normal(0, 0.1, size=(V, V))
print('vocab size:', V, '  W shape:', W.shape)

## SFT loss with prompt masking

For each pair `(prompt, response)` we compute next-token cross-entropy **only on the response tokens**. The mask is what makes this *supervised fine-tuning* and not just language modelling.

In [ ]:
def sft_loss_and_grad(W, prompt, response):
    """Cross-entropy on response tokens only; returns (loss, dW)."""
    seq = prompt + response
    ids = encode(seq)
    x = one_hot(ids[:-1])               # inputs: positions 0..T-2
    targets = ids[1:]                    # next-token labels
    logits = x @ W                       # (T-1, V)
    probs = softmax(logits)

    # Mask: 1 for positions whose *target* is inside the response, 0 for prompt.
    mask = np.zeros(len(targets))
    mask[len(prompt)-1:] = 1.0           # first response token onward
    n_resp = mask.sum()

    # Loss
    log_probs = np.log(probs[np.arange(len(targets)), targets] + 1e-12)
    loss = -(log_probs * mask).sum() / n_resp

    # Gradient: dL/dlogits = (p - one_hot(target)) * mask / n_resp
    d_logits = probs.copy()
    d_logits[np.arange(len(targets)), targets] -= 1.0
    d_logits *= mask[:, None] / n_resp
    dW = x.T @ d_logits
    return loss, dW

prompt = 'q '
response = 'hello'
loss, dW = sft_loss_and_grad(W, prompt, response)
print(f'initial loss = {loss:.3f}   dW norm = {np.linalg.norm(dW):.3f}')

## Training loop

Gradient descent on a single (prompt, response) pair until the model can complete it. The loss should fall toward zero.

In [ ]:
W = rng.normal(0, 0.1, size=(V, V))
losses = []
for step in range(400):
    loss, dW = sft_loss_and_grad(W, prompt, response)
    W -= 0.5 * dW
    losses.append(loss)

plt.plot(losses, color='#6366f1')
plt.xlabel('step'); plt.ylabel('SFT loss')
plt.title('Single-example SFT')
plt.show()

## Catastrophic forgetting

Now pretrain on a broad mix of pairs, measure how well the model handles a held-out general pair, then fine-tune *aggressively* on a single narrow pair. Watch the held-out loss rise.

In [ ]:
pretrain_pairs = [('q ', 'apple'), ('q ', 'beach'), ('q ', 'cloud'),
                  ('q ', 'dance'), ('q ', 'eagle'), ('q ', 'flame')]
held_out = ('q ', 'green')
narrow_pair = ('q ', 'zzzzz')

W = rng.normal(0, 0.1, size=(V, V))
for _ in range(800):
    p, r = pretrain_pairs[rng.integers(len(pretrain_pairs))]
    _, dW = sft_loss_and_grad(W, p, r)
    W -= 0.3 * dW

base_held = sft_loss_and_grad(W, *held_out)[0]
print(f'after pretrain   held-out loss = {base_held:.3f}')

held_curve = []
for step in range(300):
    _, dW = sft_loss_and_grad(W, *narrow_pair)
    W -= 1.5 * dW                            # aggressive LR
    held_curve.append(sft_loss_and_grad(W, *held_out)[0])

plt.plot(held_curve, color='#f43f5e')
plt.axhline(base_held, ls='--', color='#94a3b8', label='pretrain baseline')
plt.xlabel('aggressive-FT step'); plt.ylabel('held-out loss')
plt.title('Catastrophic forgetting: held-out loss rises')
plt.legend(); plt.show()

## ✏️ Your turn

Write `mix_in_general_data` that, instead of always sampling the narrow pair, mixes general pretrain pairs in with probability `p_general`. Verify that for `p_general = 0.5`, held-out loss after 300 steps stays **lower** than the pure-narrow run above.

In [ ]:
def mix_in_general_data(W_init, pretrain, narrow, held_out, steps=300, lr=1.5, p_general=0.5, seed=1):
    rng_local = np.random.default_rng(seed)
    W = W_init.copy()
    # TODO(you): each step, with probability p_general sample from `pretrain`,
    # otherwise use `narrow`. Return final held-out loss.
    return sft_loss_and_grad(W, *held_out)[0]

W_after_pretrain = rng.normal(0, 0.1, size=(V, V))
for _ in range(800):
    p, r = pretrain_pairs[rng.integers(len(pretrain_pairs))]
    _, dW = sft_loss_and_grad(W_after_pretrain, p, r)
    W_after_pretrain -= 0.3 * dW

mixed_loss = mix_in_general_data(W_after_pretrain, pretrain_pairs, narrow_pair, held_out)
print(f'mixed-data held-out loss = {mixed_loss:.3f}')
assert mixed_loss < held_curve[-1], 'should be lower than pure-narrow run'
print('passed ✓')

<details><summary>Solution</summary>

```python
def mix_in_general_data(W_init, pretrain, narrow, held_out, steps=300, lr=1.5, p_general=0.5, seed=1):
    rng_local = np.random.default_rng(seed)
    W = W_init.copy()
    for _ in range(steps):
        if rng_local.random() < p_general:
            p, r = pretrain[rng_local.integers(len(pretrain))]
        else:
            p, r = narrow
        _, dW = sft_loss_and_grad(W, p, r)
        W -= lr * dW
    return sft_loss_and_grad(W, *held_out)[0]
```

</details>